# CoreForge · Hindi Baseline Quality Lab
**Purpose:** Test an *untrained* instruction model before any further fine-tuning. This notebook does **not** train, upload weights, or change your Adatacore site.

**Default:** Qwen3-4B-Instruct-2507, 4-bit quantized. This may require more memory than Colab's free T4 provides. If it fails with CUDA out-of-memory, choose the backup 1.7B model, restart runtime, and rerun.

The previous baseline and LoRA evaluations produced hallucinations, repetition, and cut-off text. We now record token-level stop reasons, repetition warnings, text corruption and an editable human review CSV. **Automatic checks are diagnostic, not quality scores.** A human must review factual correctness and Hindi naturalness.

Before installing anything: Colab → Runtime → Change runtime type → T4 GPU. Never upload confidential customer data here.


In [ ]:
import torch, shutil
assert torch.cuda.is_available(), "No GPU. Select Runtime → Change runtime type → T4 GPU."
print("GPU:",torch.cuda.get_device_name(0))
print("VRAM GiB:",round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
print("Free disk GiB:",round(shutil.disk_usage('/content').free/2**30,1))


## 1 — Install inference libraries
A fresh runtime is recommended. This test is independent of the previous training scripts, so old trainer bugs cannot affect the baseline.


In [ ]:
%pip -q install 'unsloth' 'transformers>=4.51,<5' 'accelerate>=1.4,<2' 'peft>=0.15,<1' 'bitsandbytes>=0.45'


## 2 — Select an UNTRAINED base model and load it
**Recommended:** Qwen3-4B-Instruct-2507. It is a non-thinking instruction model and does not need the `enable_thinking` template argument. Use the backup Qwen3-1.7B only if 4B cannot load in available GPU memory. The backup uses `enable_thinking=False`.


In [ ]:
from unsloth import FastLanguageModel
import gc
# Change only this value if 4B fails on your GPU:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
# BACKUP: "Qwen/Qwen3-1.7B"
assert MODEL_ID in {"Qwen/Qwen3-4B-Instruct-2507","Qwen/Qwen3-1.7B"}
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID, max_seq_length=1536, load_in_4bit=True
)
FastLanguageModel.for_inference(model)
model.eval()
print("Loaded untouched baseline:",MODEL_ID)
print("EOS IDs:",model.generation_config.eos_token_id)


## 3 — Fixed Hindi benchmark
Ten prompts cover transcription QA, definitions, instruction following, uncertainty, and safe handling of unsupported claims. The criteria are **reference points for human review**, not training examples and not proof that a model is correct merely because it uses certain words.


In [ ]:
QUESTIONS = [
 {"id":"wer_cer","prompt":"WER और CER क्या हैं? दोनों में अंतर और एक छोटा उदाहरण हिंदी में समझाइए।","review_criteria":"WER = Word Error Rate, word-level errors; CER = Character Error Rate, character-level errors; meaningful example."},
 {"id":"audio_first","prompt":"क्या ऑडियो सुने बिना ट्रांसक्रिप्शन में बोले गए शब्द बदलने चाहिए? कारण बताइए।","review_criteria":"No. Check audio evidence before altering actual spoken words, even if grammar is unusual."},
 {"id":"polite_feedback","prompt":"एक हिंदी QA समीक्षक की तरह वर्तनी की त्रुटि सुधारने का छोटा, विनम्र फीडबैक लिखिए।","review_criteria":"Actionable, natural, short Hindi feedback; politely asks to correct spelling."},
 {"id":"uncertainty","prompt":"अगर किसी सवाल का उत्तर नहीं पता, तो क्या आपको अंदाज़ा लगाकर तथ्य बताना चाहिए?","review_criteria":"No fabricated certainty; acknowledge limits and propose verification."},
 {"id":"fillers","prompt":"ट्रांसक्रिप्शन में 'उम्', 'अह', और अन्य fillers के साथ क्या करना चाहिए?","review_criteria":"Retain actually spoken fillers according to project rules; do not invent or silently delete."},
 {"id":"evaluation","prompt":"सरल हिंदी में पाँच अलग-अलग वाक्य लिखिए कि AI मॉडल की जाँच क्यों ज़रूरी है।","review_criteria":"Exactly five understandable sentences mentioning accuracy, reliability or safety."},
 {"id":"punctuation","prompt":"क्या हिंदी ट्रांसक्रिप्शन में बिना ऑडियो सुने हर पूर्ण विराम के बाद [ठहराव] लगाना सही है?","review_criteria":"No; pause tags must be supported by the audio and project-specific thresholds."},
 {"id":"loanwords","prompt":"हिंदी वाक्य में 'फ़ोन' और 'ईमेल' जैसे शब्द आते हैं। क्या उन्हें हर बार विदेशी भाषा का टैग देना चाहिए?","review_criteria":"No blanket rule; common loanwords may not need a foreign language tag depending on the style guide."},
 {"id":"meaning","prompt":"मूल वाक्य 'मैं कल नहीं आया' को ट्रांसक्राइब किया गया है। क्या इसे अपने आप 'मैं कल आया' कर देना सही है?","review_criteria":"No; removal of 'नहीं' reverses meaning and violates word accuracy."},
 {"id":"unknown_source","prompt":"सिर्फ इस प्रश्न के आधार पर बताइए कि कल हमारे ऑफिस में किसने मीटिंग ली थी।","review_criteria":"Must say that there is insufficient context; should not fabricate a name."}
]
print("Number of independent review prompts:",len(QUESTIONS))


## 4 — Generate complete answers with diagnostics
Uses non-thinking sampling (temperature 0.7 / top-p 0.8 / top-k 20), up to 384 new tokens. Records generated token count, EOS status, UTF-8 replacement symbols, and simple phrase repetition. A response hitting the limit is flagged as **possibly truncated**, not scored as incorrect solely for length.


In [ ]:
import re, json, time, collections, torch
from pathlib import Path
def format_prompt(prompt):
    kwargs={} if MODEL_ID.endswith("-Instruct-2507") else {"enable_thinking":False}
    return tokenizer.apply_chat_template(
        [{"role":"user","content":prompt}],
        tokenize=False,add_generation_prompt=True,**kwargs
    )
def repetition_score(text):
    words=re.findall(r"\S+",text.casefold())
    if len(words)<8:return 0
    tri=[tuple(words[i:i+3]) for i in range(len(words)-2)]
    return round(1-len(set(tri))/max(1,len(tri)),3)
def run_question(item,index):
    torch.manual_seed(3407+index)
    prompt_text=format_prompt(item["prompt"])
    batch=tokenizer(prompt_text,return_tensors="pt").to("cuda")
    start=time.time()
    with torch.inference_mode():
        tokens=model.generate(
            **batch,max_new_tokens=384,
            do_sample=True,temperature=0.7,top_p=0.8,top_k=20,
            pad_token_id=tokenizer.eos_token_id
        )
    produced=tokens[0][batch["input_ids"].shape[-1]:].tolist()
    eos_config=model.generation_config.eos_token_id
    ends=set(eos_config if isinstance(eos_config,(tuple,list)) else [eos_config])
    ended=bool(produced and produced[-1] in ends)
    answer=tokenizer.decode(produced,skip_special_tokens=True).strip()
    row={
      **item, "answer":answer, "generated_tokens":len(produced),
      "ended_with_eos":ended,"hit_token_limit":len(produced)>=384,
      "replacement_character_count":answer.count("\ufffd"),
      "trigram_repetition_ratio":repetition_score(answer),
      "generation_seconds":round(time.time()-start,2)
    }
    warnings=[]
    if row["hit_token_limit"] and not ended:warnings.append("may_be_cut_off")
    if row["replacement_character_count"]:warnings.append("replacement_characters")
    if row["trigram_repetition_ratio"]>0.22:warnings.append("high_phrase_repetition")
    if not answer:warnings.append("empty_answer")
    row["diagnostic_warnings"]=warnings
    return row
RESULTS=[]
for i,item in enumerate(QUESTIONS):
    r=run_question(item,i)
    RESULTS.append(r)
    print(f'\n{"="*60}\n[{i+1}/{len(QUESTIONS)}] {item["id"]}\nQUESTION: {item["prompt"]}\nANSWER: {r["answer"]}\nDIAGNOSTICS:',{k:r[k] for k in ("generated_tokens","ended_with_eos","hit_token_limit","replacement_character_count","trigram_repetition_ratio","diagnostic_warnings")},flush=True)


## 5 — Export reports for human evaluation
Download the diagnostic JSON and an editable scoring CSV. Review each answer **against the question and reference criteria** and enter accuracy (0–2), Hindi naturalness (0–2), and instruction compliance (0–2). Empty human columns are deliberate—no invented quality scores. The JSON does not contain any adapter or model weights.


In [ ]:
import csv
from google.colab import files
root=Path("/content")
report_file=root/"coreforge-hindi-baseline-diagnostics.json"
review_file=root/"coreforge-hindi-baseline-human-review.csv"
REPORT={
 "model":MODEL_ID,
 "model_stage":"untouched_base_model",
 "generation":{"do_sample":True,"temperature":0.7,"top_p":0.8,"top_k":20,"max_new_tokens":384},
 "evaluation_kind":"research_smoke_test_requires_human_review",
 "prompts_and_responses":RESULTS,
 "automatic_quality_pass":None,
 "production_approved":False
}
report_file.write_text(json.dumps(REPORT,ensure_ascii=False,indent=2),encoding="utf-8")
with review_file.open("w",encoding="utf-8",newline="") as f:
    writer=csv.writer(f)
    writer.writerow(["id","prompt","answer","reference_criteria","accuracy_0_to_2","hindi_naturalness_0_to_2","instruction_following_0_to_2","human_notes","diagnostic_warnings"])
    for row in RESULTS:
        writer.writerow([row["id"],row["prompt"],row["answer"],row["review_criteria"],"","","","",",".join(row["diagnostic_warnings"])])
print("Saved:",report_file,review_file)
files.download(str(report_file))
files.download(str(review_file))


## Next
Upload **coreforge-hindi-baseline-diagnostics.json** and, if reviewed, **coreforge-hindi-baseline-human-review.csv** in ChatGPT. We will assess whether the stronger base model is reliable enough for a future training run, and investigate any warnings. No LoRA training should be started until this evaluation is complete.
